# Part 3 Foundation Models

This notebook visualizes the Part 3 outputs generated by `scripts/run_part3.py`.

- Segmentation: SAM2 with point prompts
- Classification: OpenCLIP vs BiomedCLIP image embeddings
- Comparison: Part 2 classical / BUSAT baselines vs Part 3 foundation models

Note: the official SAM2 release uses the Hiera checkpoint family rather than an older `ViT-B` naming.

In [ ]:
from pathlib import Path
import json
import math

import cv2
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'outputs' / 'part3').exists() and (ROOT.parent / 'outputs' / 'part3').exists():
    ROOT = ROOT.parent

PART3 = ROOT / 'outputs' / 'part3'
METRICS = PART3 / 'metrics'
OVERLAYS = PART3 / 'overlays'
ROC = PART3 / 'roc'

print('ROOT =', ROOT)
print('PART3 exists =', PART3.exists())

In [ ]:
part1 = pd.read_csv(METRICS / 'part1_sam2_measurements.csv')
part2_seg = pd.read_csv(METRICS / 'part2_sam2_manifest.csv')
comparison = pd.read_csv(METRICS / 'classification_comparison.csv')

display(part1[['sample_id', 'modality', 'prompt_source', 'sam2_iou_score', 'diameter_used_mm']])
display(part2_seg.head())
display(comparison.sort_values(['model', 'auc_mean'], ascending=[True, False]))

In [ ]:
def show_image_grid(image_paths, ncols=4, title=None, figsize=(16, 12)):
    image_paths = [Path(p) for p in image_paths if Path(p).exists()]
    if not image_paths:
        raise FileNotFoundError('No overlay images found for the requested grid.')
    nrows = math.ceil(len(image_paths) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    for ax, path in zip(axes, image_paths):
        img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(path.stem, fontsize=9)
        ax.axis('off')
    for ax in axes[len(image_paths):]:
        ax.axis('off')
    if title:
        fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    return fig

part1_overlay_paths = sorted((OVERLAYS / 'part1_sam2').glob('*.png'))
show_image_grid(part1_overlay_paths, ncols=4, title='Part 1 SAM2 Overlays', figsize=(16, 9));

In [ ]:
neg_ids = part2_seg.query('label == 0').head(6)['image_id'].tolist()
pos_ids = part2_seg.query('label == 1').head(6)['image_id'].tolist()
selected_ids = neg_ids + pos_ids
part2_overlay_paths = [OVERLAYS / 'part2_sam2' / f'{image_id}_overlay.png' for image_id in selected_ids]
show_image_grid(part2_overlay_paths, ncols=3, title='Part 2 SAM2 Overlays (6 negative + 6 positive)', figsize=(14, 16));

In [ ]:
roc_paths = [
    ROC / 'roc_compare_logreg.png',
    ROC / 'roc_compare_svm.png',
    ROC / 'roc_compare_rf.png',
]
show_image_grid(roc_paths, ncols=3, title='ROC Comparison: Part 2 vs Part 3', figsize=(18, 5));